# Qwen3-Embedding-8B — Kaggle T4×2 v1.0.0 Qualification (VERIFY)

**English:** This notebook is the **VERIFY / QUALIFICATION** notebook: a clone-and-run qualification workflow, not a link to a maintainer-hosted endpoint. It proves Kaggle Model input resolution, the Unicode quality gate, two GPU workers, bilingual semantic search, a real 1 GPU vs 2 GPU benchmark, model capability/envelope evidence, and final acceptance artifacts. Production users should run `notebooks/kaggle-t4x2-production-demo.ipynb` (USE / DEMO) in a separate fresh session.

**Tiếng Việt:** Notebook này là notebook **XÁC MINH / ĐỊNH TÍNH** (VERIFY / QUALIFICATION): luồng công việc clone-and-run để xác minh, không phải đường dẫn tới endpoint do người duy trì vận hành. Notebook chứng minh phân giải Kaggle Model input, cổng chặn chất lượng Unicode, hai GPU worker, tìm kiếm ngữ nghĩa song ngữ, benchmark thật 1 GPU so với 2 GPU, bằng chứng năng lực/đường bao mô hình, và artifact nghiệm thu cuối. Người dùng production nên chạy `notebooks/kaggle-t4x2-production-demo.ipynb` (USE / DEMO) trong một phiên riêng.

```text
Recommended / Khuyến nghị: this notebook = VERIFY / QUALIFICATION
Use / Sử dụng production: notebooks/kaggle-t4x2-production-demo.ipynb (USE / DEMO)
GPU: T4×2
Internet: ON for Git clone
Kaggle inputs: attach the Kaggle Model and the Kaggle Dataset
Run: Restart Session → Run All
```

## Stage 0 — preflight/static identity

**English:** Clone the official repository at an exact Git ref (v1.0.0 by default, or a maintainer-only staging SHA via `REPO_REF_OVERRIDE`), record GPU inventory, keep model/data offline under Kaggle Input, and verify the CPU/static contract before spending GPU time. The normal path keeps `EXPOSE_MODE=off`.

**Tiếng Việt:** Sao chép kho chính thức tại một Git ref chính xác (mặc định v1.0.0, hoặc staging SHA chỉ dành cho người duy trì qua `REPO_REF_OVERRIDE`), ghi nhận GPU inventory, giữ model/dữ liệu offline trong Kaggle Input, và xác minh hợp đồng CPU/static trước khi dùng GPU. Đường dẫn bình thường giữ `EXPOSE_MODE=off`.

In [ ]:
import os, sys, json, subprocess, shutil
from pathlib import Path

REPO_URL = "https://github.com/dangkhoa2016/Qwen3-Embedding-8B-On-T4x2-GPU.git"
REPO_REF = "v1.0.0"
REPO_REF_OVERRIDE = ""  # maintainer-only: exact staging SHA during prepublication acceptance
CHECKOUT = Path('/kaggle/working/qwen3-embedding-8b-t4x2')
SELECTED_REF = REPO_REF_OVERRIDE.strip() or REPO_REF


def _run(*args, cwd=None, capture=False):
    return subprocess.run(list(args), cwd=cwd, check=True, text=True, capture_output=capture)


if (CHECKOUT / '.git').is_dir():
    actual_origin = _run('git', 'remote', 'get-url', 'origin', cwd=CHECKOUT, capture=True).stdout.strip()
    if (actual_origin.rstrip('/') != REPO_URL.rstrip('/').removesuffix('.git')
            and actual_origin.rstrip('/') != REPO_URL.rstrip('/')):
        _run('git', 'remote', 'set-url', 'origin', REPO_URL, cwd=CHECKOUT)
    _run('git', 'fetch', '--prune', 'origin',
         '+refs/heads/*:refs/remotes/origin/*', '+refs/tags/*:refs/tags/*', cwd=CHECKOUT)
else:
    if CHECKOUT.exists():
        shutil.rmtree(CHECKOUT)
    _run('git', 'clone', '--no-checkout', REPO_URL, str(CHECKOUT))
    _run('git', 'fetch', '--prune', 'origin',
         '+refs/heads/*:refs/remotes/origin/*', '+refs/tags/*:refs/tags/*', cwd=CHECKOUT)

_run('git', 'checkout', '--detach', '--force', SELECTED_REF, cwd=CHECKOUT)
head = _run('git', 'rev-parse', 'HEAD', cwd=CHECKOUT, capture=True).stdout.strip()
status = _run('git', 'status', '--porcelain', cwd=CHECKOUT, capture=True).stdout
if status:
    raise RuntimeError(f'checkout is dirty: {status}')
PROJECT_ROOT = CHECKOUT.resolve()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
os.environ.update({
    'HF_HUB_OFFLINE': '1', 'TRANSFORMERS_OFFLINE': '1', 'HF_DATASETS_OFFLINE': '1',
    'TOKENIZERS_PARALLELISM': 'false', 'EXPOSE_MODE': 'off',
})
print('SELECTED_REF=', SELECTED_REF)
print('GIT_HEAD=', head)
print('PROJECT_ROOT=', PROJECT_ROOT)

def bootstrap_and_adopt_environment(project_root: Path) -> dict[str, str]:
    shell_command = (
        "source kaggle/bootstrap.sh 1>&2\n"
        "env -0\n"
    )

    proc = subprocess.run(
        ["bash", "-c", shell_command],
        cwd=str(project_root),
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    bootstrap_log = proc.stderr.decode("utf-8", errors="replace")

    if bootstrap_log:
        print(bootstrap_log[-6000:])

    if proc.returncode != 0:
        raise RuntimeError(f"bootstrap failed with rc={proc.returncode}")

    exported: dict[str, str] = {}

    for item in proc.stdout.split(b"\0"):
        if not item:
            continue

        key, sep, value = item.partition(b"=")

        if not sep:
            continue

        exported[
            key.decode("utf-8", errors="strict")
        ] = value.decode("utf-8", errors="surrogateescape")

    for key in ("PATH", "PYTHONPATH"):
        if not exported.get(key):
            raise RuntimeError(f"bootstrap environment missing {key}")

    for key in ("PATH", "PYTHONPATH", "LD_LIBRARY_PATH"):
        if key in exported:
            os.environ[key] = exported[key]

    return {
        key: os.environ[key]
        for key in ("PATH", "PYTHONPATH", "LD_LIBRARY_PATH")
        if key in os.environ
    }

BOOTSTRAP_ENV = bootstrap_and_adopt_environment(PROJECT_ROOT)
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)
from app.model_resolver import resolve_kaggle_model_dir
print('Kaggle Model input:', resolve_kaggle_model_dir(Path('/kaggle/input'), os.environ.get('KAGGLE_MODEL_DIR')))
print('Unicode quality gate: covered by the full regression suite above')

## Stage 1 — 5K acceptance

**English:** A 5K run is a functional smoke only. It force-builds a separate index, starts one and then two GPU workers, exercises semantic search, and records the 1 GPU vs 2 GPU benchmark. It must finish with `FINAL_ACCEPTANCE=PASS`.

**Tiếng Việt:** Lần chạy 5K chỉ là smoke kiểm tra chức năng. Nó ép xây một index riêng, khởi động một rồi hai GPU worker, kiểm tra tìm kiếm ngữ nghĩa, và ghi lại benchmark 1 GPU so với 2 GPU. Bắt buộc kết thúc với `FINAL_ACCEPTANCE=PASS`.

In [ ]:
WORK5 = Path('/kaggle/working/qwen3-embedding-8b-t4x2-v013-runtime-5k')
env5 = os.environ.copy()
env5.update({'WORKING_DIR':str(WORK5), 'DEMO_CORPUS_LIMIT':'5000', 'FORCE_REBUILD_INDEX':'1', 'EXPOSE_MODE':'off'})
subprocess.run(['bash','kaggle/acceptance.sh'], env=env5, check=True)
ACCEPTANCE_5K = sorted(WORK5.glob('evidence-*'))[-1]
print('ACCEPTANCE_5K=', ACCEPTANCE_5K)


## Stage 2 — forced-fresh 100K acceptance

**English:** The canonical run removes the corpus limit and forces a fresh 100K index using only source-controlled batching/allocator defaults. No external tunnel is involved. This is the release gate for the historical 100K padding/OOM regression.

**Tiếng Việt:** Lần chạy chuẩn bỏ giới hạn corpus và ép xây index 100K mới chỉ dùng mặc định batching/allocator trong mã nguồn. Không dùng tunnel ngoài. Đây là cổng phát hành cho hồi quy padding/OOM 100K trong quá khứ.

In [ ]:
WORK100 = Path('/kaggle/working/qwen3-embedding-8b-t4x2-v013-runtime-100k')
env100 = os.environ.copy()
env100.pop('DEMO_CORPUS_LIMIT', None)
env100.update({'WORKING_DIR':str(WORK100), 'FORCE_REBUILD_INDEX':'1', 'EXPOSE_MODE':'off'})
subprocess.run(['bash','kaggle/acceptance.sh'], env=env100, check=True)
ACCEPTANCE_100K = sorted(WORK100.glob('evidence-*'))[-1]
print('ACCEPTANCE_100K=', ACCEPTANCE_100K)


## Stage 3 — semantic/model capability qualification

**English:** Run the frozen 20-case semantic suite, MRL dimension sweep, instruction A/B, EN/VI cross-lingual cases, code-retrieval fixture, and dataset EN/VI quality measurement against the proven 100K index.

**Tiếng Việt:** Chạy bộ 20-case ngữ nghĩa cố định, quét chiều MRL, so sánh A/B theo lệnh, các ca xuyên ngôn ngữ EN/VI, fixture truy xuất mã, và đo chất lượng dataset EN/VI trên index 100K đã được kiểm chứng.

In [ ]:
QUAL_WORK = Path('/kaggle/working/qwen3-embedding-8b-t4x2-v013-qualification')
ENABLE_CLOUDFLARE = 0  # Keep disabled for normal portfolio qualification.
subprocess.run([
    'bash','kaggle/qualification.sh',
    '--work-root',str(QUAL_WORK),
    '--acceptance-5k',str(ACCEPTANCE_5K),
    '--acceptance-100k',str(ACCEPTANCE_100K),
    '--enable-cloudflare',str(ENABLE_CLOUDFLARE),
], check=True)
QUAL_DIRS = sorted(p for p in QUAL_WORK.glob('qualification-*') if p.is_dir())
if not QUAL_DIRS:
    raise RuntimeError('qualification evidence directory not found')
QUAL_DIR = QUAL_DIRS[-1]
print(json.dumps(json.loads((QUAL_DIR/'semantic-quality.json').read_text()), indent=2)[:6000])


## Stage 4 — hardware/long-context envelope

**English:** The same qualification archive contains the curated 1/2-worker saturation matrix and isolated 4K→32K long-context probe. Envelope OOM is measured evidence; it does not weaken the normal 4096-token final acceptance contract.

**Tiếng Việt:** Cùng một kho lưu trữ định tính chứa ma trận bão hòa 1/2 worker được tuyển chọn và phép dò long-context cô lập 4K→32K. OOM đường bao là bằng chứng đo được; nó không làm yếu hợp đồng nghiệm thu cuối 4096-token thông thường.

In [ ]:
for name in ('hardware-envelope.json', 'long-context-envelope.json', 'dataset-quality.json'):
    print()
    print('===', name, '===')
    print((QUAL_DIR/name).read_text()[:6000])


## Stage 5 — optional authenticated Cloudflare external demo

**English:** Cloudflare Quick Tunnel is **optional**. Do not publish the generated hostname. To prove external serving in your own session, rerun the qualification command with `ENABLE_CLOUDFLARE = 1`; it restarts the dual service with an ephemeral Bearer key, checks unauthorized/authorized remote behavior, scans retained evidence for the secret, and tears the tunnel down. This project is not a permanently hosted public inference endpoint.

**Tiếng Việt:** Cloudflare Quick Tunnel là **tùy chọn**. Không công bố hostname được tạo. Để chứng minh phục vụ từ bên ngoài trong phiên của bạn, chạy lại lệnh định tính với `ENABLE_CLOUDFLARE = 1`; lệnh khởi động lại dịch vụ kép với Bearer key tạm thời, kiểm tra hành vi từ xa không/được phép, quét bằng chứng đã lưu để tìm bí mật, rồi gỡ tunnel. Dự án này không phải là endpoint inference công khai vận hành thường xuyên.

In [ ]:
# Intentionally NOT executed by default.
# ENABLE_CLOUDFLARE = 1
# subprocess.run([
#     'bash','kaggle/qualification.sh', '--work-root',str(QUAL_WORK),
#     '--acceptance-5k',str(ACCEPTANCE_5K), '--acceptance-100k',str(ACCEPTANCE_100K),
#     '--enable-cloudflare','1'
# ], check=True)


## Stage 6 — evidence archive verification

**English:** Verify final acceptance/qualification checksums before using any measured number in README, Kaggle descriptions, LinkedIn, or other portfolio material.

**Tiếng Việt:** Xác minh checksum nghiệm thu/định tính cuối trước khi dùng bất kỳ con số đo nào trong README, mô tả Kaggle, LinkedIn hoặc tài liệu portfolio khác.

In [ ]:
def verify_sha_file(path: Path):
    subprocess.run(['sha256sum','-c',path.name], cwd=path.parent, check=True)

for sha in sorted(WORK5.glob('*.tar.gz.sha256'))[-1:] + sorted(WORK100.glob('*.tar.gz.sha256'))[-1:] + sorted(QUAL_WORK.glob('*.tar.gz.sha256'))[-1:]:
    print('verifying', sha)
    verify_sha_file(sha)
print('final acceptance + qualification evidence verification: PASS')
